In [ ]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git

Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 27360, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 27360 (delta 69), reused 21 (delta 20), pack-reused 27220 (from 3)
Receiving objects: 100% (27360/27360), 13.24 MiB | 15.97 MiB/s, done.
Resolving deltas: 100% (19585/19585), done.


In [ ]:
!pip install "torch>=2.6.0" torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
!pip install \
  transformers>=4.51.0 \
  accelerate==1.11.0 \
  bitsandbytes==0.46.1 \
  peft==0.18.1 \
  trl==0.24.0 \
  datasets -q

In [ ]:
!pip install -e "/content/LLaMA-Factory[metrics]" --no-deps -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llamafactory (pyproject.toml) ... done


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

import zipfile

zip_path = "/content/drive/MyDrive/img-captioning/vlm-4class.zip"
extract_path = "/content/dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"Ekstraksi selesai! Dataset siap di {extract_path}")

Mounted at /content/drive
Ekstraksi selesai! Dataset siap di /content/dataset


In [ ]:
!mv /content/dataset/vlm/* /content/dataset/
!rm -rf /content/dataset/vlm/

In [ ]:
import json, os

file_path = "/content/dataset/adas_caption_train.json"
base_image_dir = "/content/dataset"

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

fixed_data = []
skipped = 0

role_map = {"human": "user", "gpt": "assistant"}

for item in data:
    # Convert "image": "path.jpg" → "images": ["/absolute/path.jpg"]
    if "image" in item:
        img_path = item.pop("image")
        abs_path = os.path.join(base_image_dir, img_path)
        item["images"] = [abs_path]
    elif "images" in item:
        item["images"] = [
            img if img.startswith("/") else os.path.join(base_image_dir, img)
            for img in (
                [item["images"]] if isinstance(item["images"], str) else item["images"]
            )
        ]

    # Normalize roles: human→user, gpt→assistant
    for conv in item.get("conversations", []):
        conv["from"] = role_map.get(conv["from"], conv["from"])

    # Validate
    if not item.get("images") or not item.get("conversations"):
        skipped += 1
        continue

    fixed_data.append(item)

print(f"Fixed: {len(fixed_data)} | Skipped: {skipped}")
print(f"\nSample output:\n{json.dumps(fixed_data[0], indent=2)}")

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(fixed_data, f, ensure_ascii=False, indent=2)

Fixed: 16710 | Skipped: 0

Sample output:
{
  "id": "pothole_0_v1",
  "conversations": [
    {
      "from": "user",
      "value": "<image>\nLakukan analisis pada gambar ini dan sebutkan jenis kerusakan beserta tingkat keparahannya."
    },
    {
      "from": "assistant",
      "value": "Terdapat retak melintang yang tergolong ringan."
    }
  ],
  "images": [
    "/content/dataset/cropped_images/crop_0_AA_10800_jpeg.rf.0ffd63a44a4e8d68df26630ad4caa019.jpg"
  ]
}


In [ ]:
from sklearn.model_selection import train_test_split

# ── Split 80 / 10 / 10 ──────────────────────────────────────────────────────
train_data, temp_data = train_test_split(fixed_data, test_size=0.2, random_state=42)
val_data,   test_data = train_test_split(temp_data,  test_size=0.5, random_state=42)

for split_name, split_data, fname in [
    ("Train", train_data, "qwen_train.json"),
    ("Val",   val_data,   "qwen_val.json"),
    ("Test",  test_data,  "qwen_test.json"),
]:
    out_path = f"/content/dataset/{fname}"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(split_data, f, ensure_ascii=False, indent=2)
    print(f"Data {split_name}: {len(split_data)} sampel → {out_path}")

Data Train: 13368 sampel → /content/dataset/qwen_train.json
Data Val: 1671 sampel → /content/dataset/qwen_val.json
Data Test: 1671 sampel → /content/dataset/qwen_test.json


In [ ]:
info_path = "/content/LLaMA-Factory/data/dataset_info.json"

new_datasets = {
    "adas_qwen_train": {
        "file_name": "/content/dataset/qwen_train.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_role": "user",         # ← fixed
            "bot_role": "assistant"      # ← fixed
        }
    },
    "adas_qwen_val": {
        "file_name": "/content/dataset/qwen_val.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "images": "images"},
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_role": "user",         # ← fixed
            "bot_role": "assistant"      # ← fixed
        }
    }
}

with open(info_path, "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

dataset_info.update(new_datasets)

with open(info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2)

print("\nDataset Train dan Val berhasil didaftarkan!")


Dataset Train dan Val berhasil didaftarkan!


In [ ]:
import json, os

info_path = "/content/LLaMA-Factory/data/dataset_info.json"

with open(info_path, "r") as f:
    dataset_info = json.load(f)

for key in ["adas_qwen_train", "adas_qwen_val"]:
    dataset_info[key]["tags"] = {
        "role_tag": "from",
        "content_tag": "value",
        "user_tag": "user",       
        "assistant_tag": "assistant" 
    }

with open(info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

with open("/content/dataset/qwen_train.json") as f:
    data = json.load(f)

roles_found = set()
for item in data:
    for conv in item["conversations"]:
        roles_found.add(conv["from"])

print(f"Roles in JSON: {roles_found}")
print(f"Expected:      {{'user', 'assistant'}}")
print(f"Match: {roles_found == {'user', 'assistant'}}")
print(f"\ndataset_info tags: {dataset_info['adas_qwen_train']['tags']}")

Roles in JSON: {'user', 'assistant'}
Expected:      {'user', 'assistant'}
Match: True

dataset_info tags: {'role_tag': 'from', 'content_tag': 'value', 'user_tag': 'user', 'assistant_tag': 'assistant'}


In [ ]:
# Permanently patch nn.Module to add set_submodule
import torch.nn as nn

def _set_submodule(self, target: str, module: nn.Module):
    parts = target.split(".")
    parent = self
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], module)

# Inject into nn.Module class
nn.Module.set_submodule = _set_submodule
print("✓ nn.Module.set_submodule patched!")

# Verify
import torch
m = torch.nn.Linear(4, 4)
test = torch.nn.Sequential(m)
test.set_submodule("0", torch.nn.Linear(8, 8))
print("✓ Patch works correctly!")

✓ nn.Module.set_submodule patched!
✓ Patch works correctly!


In [ ]:
bnb_path = "/usr/local/lib/python3.12/dist-packages/transformers/integrations/bitsandbytes.py"

with open(bnb_path, "r") as f:
    lines = f.readlines()

# Fix line 222 (index 221): replace set_submodule with workaround
# Fix lines 226-228 (index 225-227): close the unclosed logger.warning

# Fix 1: patch set_submodule (line 222, index 221)
lines[221] = '                    # patched set_submodule workaround\n'
lines.insert(222, '                    _parts = module_name.split(".")\n')
lines.insert(223, '                    _parent = model\n')
lines.insert(224, '                    for _part in _parts[:-1]:\n')
lines.insert(225, '                        _parent = getattr(_parent, _part)\n')
lines.insert(226, '                    setattr(_parent, _parts[-1], new_module)\n')

with open(bnb_path, "w") as f:
    f.writelines(lines)

with open(bnb_path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines[225:240], start=226):
    print(f"{i}: {repr(line)}")

226: '                        _parent = getattr(_parent, _part)\n'
227: '                    setattr(_parent, _parts[-1], new_module)\n'
228: '                    has_been_replaced = True\n'
229: '\n'
230: '    if not has_been_replaced:\n'
231: '        logger.warning(\n'
232: '            "You are loading your model using eetq but no linear modules were found in your model."\n'
233: '            " Please double check your model architecture, or submit an issue on github if you think this is"\n'
234: '            " a bug."\n'
235: '        )\n'
236: '    return model\n'
237: '\n'
238: '\n'
239: '# Copied from PEFT: https://github.com/huggingface/peft/blob/47b3712898539569c02ec5b3ed4a6c36811331a1/src/peft/utils/integrations.py#L41\n'
240: 'def dequantize_bnb_weight(weight: "torch.nn.Parameter", state=None):\n'


In [ ]:
bnb_path = "/usr/local/lib/python3.12/dist-packages/transformers/integrations/bitsandbytes.py"

with open(bnb_path, "r") as f:
    content = f.read()

# Find and replace the entire broken block from set_submodule to end of function
import re

# Replace the broken section wholesale
old = re.search(
    r'(                    # patched set_submodule workaround.*?)'
    r'(# Copied from PEFT)',
    content, re.DOTALL
)

if old:
    print("Found block:")
    print(repr(old.group(1)))
else:
    # Show 50 chars around the problem
    idx = content.find("logger.warning(\n")
    while idx != -1:
        snippet = content[idx:idx+200]
        if "def " in snippet and ")" not in snippet.split("def")[0]:
            print(f"Unclosed at index {idx}:")
            print(repr(snippet))
        idx = content.find("logger.warning(\n", idx+1)

Found block:
'                    # patched set_submodule workaround\n                    _parts = module_name.split(".")\n                    _parent = model\n                    for _part in _parts[:-1]:\n                        _parent = getattr(_parent, _part)\n                    setattr(_parent, _parts[-1], new_module)\n                    has_been_replaced = True\n\n    if not has_been_replaced:\n        logger.warning(\n            "You are loading your model using eetq but no linear modules were found in your model."\n            " Please double check your model architecture, or submit an issue on github if you think this is"\n            " a bug."\n        )\n    return model\n\n\n'


In [ ]:
bnb_path = "/usr/local/lib/python3.12/dist-packages/transformers/integrations/bitsandbytes.py"

with open(bnb_path, "r") as f:
    lines = f.readlines()

# Rewrite lines 231-236 (index 230-235) as one clean warning call
new_block = [
    '    if not has_been_replaced:\n',
    '        logger.warning(\n',
    '            "You are loading your model using eetq but no linear modules were found in your model."\n',
    '            " Please double check your model architecture, or submit an issue on github if you think this is"\n',
    '            " a bug."\n',
    '        )\n',
]

# Replace lines 230-236 (index 229-235)
lines[229:236] = new_block

with open(bnb_path, "w") as f:
    f.writelines(lines)

import py_compile
try:
    py_compile.compile(bnb_path, doraise=True)
    print("✓ Fixed!")
    # Show final result
    with open(bnb_path) as f:
        ls = f.readlines()
    for i, l in enumerate(ls[219:240], start=220):
        print(f"{i}: {repr(l)}")
except py_compile.PyCompileError as e:
    print(f"✗ {e}")

✓ Fixed!
220: '                    # Force requires grad to False to avoid unexpected errors\n'
221: '                    new_module.requires_grad_(False)\n'
222: '                    # patched set_submodule workaround\n'
223: '                    _parts = module_name.split(".")\n'
224: '                    _parent = model\n'
225: '                    for _part in _parts[:-1]:\n'
226: '                        _parent = getattr(_parent, _part)\n'
227: '                    setattr(_parent, _parts[-1], new_module)\n'
228: '                    has_been_replaced = True\n'
229: '\n'
230: '    if not has_been_replaced:\n'
231: '        logger.warning(\n'
232: '            "You are loading your model using eetq but no linear modules were found in your model."\n'
233: '            " Please double check your model architecture, or submit an issue on github if you think this is"\n'
234: '            " a bug."\n'
235: '        )\n'
236: '\n'
237: '\n'
238: '# Copied from PEFT: https://github.com/h

In [ ]:
module_path = "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py"

with open(module_path, "r") as f:
    content = f.read()

# Add set_submodule method right after get_submodule
patch = '''
    def set_submodule(self, target: str, module: "Module") -> None:
        """Set a submodule given a target string (patched)."""
        parts = target.split(".")
        parent = self
        for part in parts[:-1]:
            parent = getattr(parent, part)
        setattr(parent, parts[-1], module)

'''

# Insert after get_submodule definition
marker = "    def get_submodule("
if "def set_submodule(" not in content:
    idx = content.find(marker)
    if idx != -1:
        # Find end of get_submodule method by finding next def
        next_def = content.find("\n    def ", idx + 1)
        content = content[:next_def] + patch + content[next_def:]
        with open(module_path, "w") as f:
            f.write(content)

        import py_compile
        py_compile.compile(module_path, doraise=True)
        print("✓ torch nn.Module patched at source!")
    else:
        print("Marker not found")
else:
    print("set_submodule already exists in torch — checking why it fails...")
    for i, line in enumerate(content.split("\n")):
        if "set_submodule" in line:
            print(f"  line {i}: {line}")

set_submodule already exists in torch — checking why it fails...
  line 738:     def set_submodule(
  line 769:         could call ``set_submodule("net_b.net_c.conv", nn.Linear(1, 1))``
  line 773:         you would call ``set_submodule("net_b.conv", nn.Conv2d(1, 1, 1))``.
  line 776:         ``set_submodule("net_b.conv", nn.Conv2d(1, 1, 1), strict=True)``, an AttributeError


In [ ]:
import py_compile, subprocess, sys

# Verify both files are clean
for path in [
    "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py",
    "/usr/local/lib/python3.12/dist-packages/transformers/integrations/bitsandbytes.py"
]:
    try:
        py_compile.compile(path, doraise=True)
        print(f"✓ {path.split('/')[-1]}")
    except Exception as e:
        print(f"✗ {path.split('/')[-1]}: {e}")

# Verify set_submodule now exists
result = subprocess.run(
    [sys.executable, "-c",
     "import torch.nn as nn; m=nn.Linear(2,2); print(hasattr(m,'set_submodule'))"],
    capture_output=True, text=True
)
print(f"set_submodule exists: {result.stdout.strip()}")

✓ module.py
✓ bitsandbytes.py
set_submodule exists: True


In [ ]:
visual_path = "/content/LLaMA-Factory/src/llamafactory/model/model_utils/visual.py"

with open(visual_path, "r") as f:
    content = f.read()

old = '''    model_type="qwen2_vl",
    projector_keys=["visual.merger"],
    vision_model_keys=["visual.patch_embed", "visual.blocks"],'''

new = '''    model_type="qwen2_vl",
    projector_keys=["model.visual.merger"],
    vision_model_keys=["model.visual.patch_embed", "model.visual.blocks"],'''

if old in content:
    content = content.replace(old, new)
    with open(visual_path, "w") as f:
        f.write(content)
    print("✓ Patched!")
else:
    print("Still not found — showing full qwen2_vl block:")
    idx = content.find('"qwen2_vl"')
    print(content[idx-50:idx+300])

✓ Patched!


In [ ]:
import os, glob, subprocess, sys
import torch.nn as nn

def _set_submodule(self, target, module):
    parts = target.split(".")
    parent = self
    for part in parts[:-1]:
        parent = getattr(parent, part)
    setattr(parent, parts[-1], module)
nn.Module.set_submodule = _set_submodule

os.environ["WANDB_DISABLED"] = "true"

output_dir = "/content/drive/MyDrive/img-captioning/vlm/qwen2-2b-4class"

# Create output dir if it doesn't exist yet
os.makedirs(output_dir, exist_ok=True)
print(f"✓ Output dir ready: {output_dir}")

# Remove scaler files from all checkpoints
checkpoints_found = glob.glob(f"{output_dir}/checkpoint-*")
for ckpt in checkpoints_found:
    scaler_file = os.path.join(ckpt, "scaler.pt")
    if os.path.exists(scaler_file):
        os.remove(scaler_file)
        print(f"✓ Removed scaler: {os.path.basename(ckpt)}")

# Find latest checkpoint
numeric_ckpts = []
for ck in checkpoints_found:
    try:
        step = int(os.path.basename(ck).replace("checkpoint-", "").split("-")[0])
        numeric_ckpts.append((step, ck))
    except ValueError:
        pass

numeric_ckpts.sort(key=lambda x: x[0])
if numeric_ckpts:
    latest_step, latest_ckpt = numeric_ckpts[-1]
    print(f"✓ Resuming from step {latest_step}: {latest_ckpt}")
else:
    latest_ckpt = None
    print("No checkpoint found, starting fresh.")

cmd = [
    "llamafactory-cli", "train",
    "--stage", "sft",
    "--do_train", "True",
    "--model_name_or_path", "Qwen/Qwen2-VL-2B-Instruct",
    "--dataset", "adas_qwen_train",      
    "--eval_dataset", "adas_qwen_val",   
    "--template", "qwen2_vl",
    "--finetuning_type", "lora",
    "--lora_target", "all",
    "--lora_rank", "8",
    "--dataset_dir", "/content/LLaMA-Factory/data",
    "--output_dir", output_dir,
    "--overwrite_cache", "True",
    "--overwrite_output_dir", "False",
    "--cutoff_len", "1024",
    "--per_device_train_batch_size", "1",
    "--gradient_accumulation_steps", "4",
    "--lr_scheduler_type", "cosine",
    "--logging_steps", "10",
    "--eval_strategy", "steps",
    "--eval_steps", "100",
    "--save_strategy", "steps",
    "--save_steps", "200",
    "--learning_rate", "5e-5",
    "--num_train_epochs", "10",
    "--quantization_bit", "4",
    "--bf16", "True",
    "--load_best_model_at_end", "True",
    "--metric_for_best_model", "eval_loss",
]

if latest_ckpt:
    cmd += ["--resume_from_checkpoint", latest_ckpt]

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd="/content/LLaMA-Factory"
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()
print(f"\nExit code: {process.returncode}")

✓ Output dir ready: /content/drive/MyDrive/img-captioning/vlm/qwen2-2b-4class
✓ Resuming from step 33000: /content/drive/MyDrive/img-captioning/vlm/qwen2-2b-4class/checkpoint-33000
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
[WARNING|2026-05-26 16:21:25] llamafactory.hparams.parser:149 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-05-26 16:21:25] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configurati